# 13. 영상 이진화

전역 임계값 모드와 평균·가우시안 적응형 이진화를 비교합니다.

> 문서 예제 이미지는 노트북 옆 `data` 폴더에 `crossword.jpg`로 넣으세요.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_image(name, flags=cv2.IMREAD_COLOR):
    path = Path("data") / name
    image = cv2.imread(str(path), flags)
    if image is None:
        raise FileNotFoundError(path)
    return image

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 4))
    axes = np.atleast_1d(axes)
    for ax, image, title in zip(axes, images, titles):
        if image.ndim == 2:
            ax.imshow(image, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()


## 임계값 모드 비교

In [ ]:
gradient = np.tile(np.arange(256, dtype=np.uint8), (180, 1))
modes = [
    (cv2.THRESH_BINARY, "BINARY"),
    (cv2.THRESH_BINARY_INV, "BINARY_INV"),
    (cv2.THRESH_TRUNC, "TRUNC"),
    (cv2.THRESH_TOZERO, "TOZERO"),
    (cv2.THRESH_TOZERO_INV, "TOZERO_INV"),
]
results, titles = [gradient], ["source"]
for mode, name in modes:
    _, dst = cv2.threshold(gradient, 127, 255, mode)
    results.append(dst)
    titles.append(name)
show(results, titles)


## 전역 임계값

In [ ]:
document = read_image("crossword.jpg", cv2.IMREAD_GRAYSCALE)
_, global_binary = cv2.threshold(document, 127, 255, cv2.THRESH_BINARY)
show([document, global_binary], ["source", "global threshold"])


## 평균 적응형 이진화

In [ ]:
adaptive_mean = cv2.adaptiveThreshold(
    document, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY, blockSize=21, C=10,
)
show([document, global_binary, adaptive_mean], ["source", "global", "adaptive mean"])


## 가우시안 적응형 이진화

In [ ]:
adaptive_gaussian = cv2.adaptiveThreshold(
    document, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, blockSize=21, C=10,
)
comparison = cv2.addWeighted(adaptive_mean, 0.5, adaptive_gaussian, 0.5, 0)
show([adaptive_mean, adaptive_gaussian, comparison], ["adaptive mean", "adaptive Gaussian", "overlay"])


## blockSize와 C 비교

In [ ]:
settings = [(11, 5), (21, 10), (41, 15)]
outputs = [cv2.adaptiveThreshold(document, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block, c) for block, c in settings]
show(outputs, [f"block={block}, C={c}" for block, c in settings])
